# Gaffer · v0.5 Phase 1 — Homography validation (the trust gate)

Goal: turn hand-clicked calibration points into a **trustworthy H matrix** that
maps image pixels → pitch metres. The milestone is NOT the minimap — it's an H
whose **reprojection error < 10 px**. If the gate fails, fix the points before
doing anything downstream; a bad H poisons every later analytic.

**Prerequisite:** run the click collector first to produce the calibration JSON:
```bash
uv run python scripts/collect_calibration.py data/test_clips/tactical_playlist_1.mp4 --time 65
```


In [ ]:
import json
from pathlib import Path
import cv2, numpy as np
import matplotlib.pyplot as plt

from gaffer import config
from gaffer.calibration.pitch_model import PitchModel
from gaffer.calibration.homography import HomographyEstimator
from gaffer.video.loader import VideoLoader

CLIP = 'data/test_clips/tactical_playlist_1.mp4'
CALIB = config.DATA_DIR / 'calibration' / (Path(CLIP).stem + '.json')
print('calibration file:', CALIB, '· exists:', CALIB.exists())

def show(img, title='', figsize=(12, 7)):
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); plt.title(title); plt.axis('off'); plt.show()

## 1 · Load calibration + build correspondences
World coordinates come from `config.PITCH_KEYPOINTS`; the JSON only stores the
clicked pixel positions.


In [ ]:
data = json.loads(CALIB.read_text())
img_pts_named = data['image_points']      # {name: [px, py]}
frame_idx = data['frame_idx']

names = list(img_pts_named.keys())
image_pts = np.array([img_pts_named[n] for n in names], dtype=np.float32)
world_pts = np.array([config.PITCH_KEYPOINTS[n] for n in names], dtype=np.float32)

print(f'{len(names)} correspondences from frame {frame_idx}:')
for n in names:
    print(f'  {n:<22} px={img_pts_named[n]}  ->  world_m={config.PITCH_KEYPOINTS[n]}')

## 2 · Compute H  (pixels → metres)


In [ ]:
est = HomographyEstimator()
H, valid = est.compute(image_pts, world_pts)
print('valid:', valid)
print('H=\n', H)

## 3 · ⚠️ THE GATE — reprojection error < 10 px
Project the world landmarks back into the image via H⁻¹ and compare to the
clicked pixels. **If mean error >= 10 px, STOP and re-collect points.**


In [ ]:
mean_err, max_err, errs = est.reprojection_error(image_pts, world_pts, H)
print(f'mean reprojection error: {mean_err:.2f} px')
print(f'max  reprojection error: {max_err:.2f} px')
for n, e in zip(names, errs):
    flag = '  <-- worst offender' if e == max_err else ''
    print(f'  {n:<22} {e:6.2f} px{flag}')

assert mean_err < 10.0, (
    f'GATE FAILED: mean reprojection {mean_err:.1f}px >= 10px. '
    'Re-run collect_calibration.py with more careful / better-spread points '
    '(drop the worst offender above). Do NOT proceed to the minimap.')
print('\nGATE PASSED — H is trustworthy.')

## 4 · View A — clicked points on the source frame


In [ ]:
with VideoLoader(CLIP) as v:
    v.seek(frame_idx); ok, frame = v.read()
viewA = frame.copy()
for n in names:
    px, py = img_pts_named[n]
    cv2.circle(viewA, (px, py), 6, (0, 255, 255), -1)
    cv2.putText(viewA, n, (px+6, py-6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1, cv2.LINE_AA)
show(viewA, 'View A — calibration points on source frame')

## 5 · View B — same landmarks on the pitch template
Confirms the world coordinates land where they should on the 2D pitch.


In [ ]:
pm = PitchModel()
viewB = pm.draw_pitch()
for n in names:
    xm, ym = config.PITCH_KEYPOINTS[n]
    cx, cy = pm.pitch_to_pixels(xm, ym)
    cv2.circle(viewB, (cx, cy), 6, (0, 255, 255), -1)
    cv2.putText(viewB, n, (cx+6, cy-6), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0,255,255), 1, cv2.LINE_AA)
show(viewB, 'View B — landmarks on pitch template', figsize=(11, 8))

## 6 · View C — projection sanity check
Project the clicked image points through H → metres. They should land on their
known landmark positions (error ≈ the reprojection error above). This confirms
the forward mapping (px → m) that the minimap will use.


In [ ]:
viewC = pm.draw_pitch()
for n in names:
    xm, ym = est.project(img_pts_named[n], H)        # projected (metres)
    cx, cy = pm.pitch_to_pixels(xm, ym)
    gx, gy = pm.pitch_to_pixels(*config.PITCH_KEYPOINTS[n])  # ground truth
    cv2.circle(viewC, (gx, gy), 7, (0, 255, 0), 2)           # green = truth
    cv2.circle(viewC, (cx, cy), 4, (0, 0, 255), -1)          # red = projected
show(viewC, 'View C — projected (red) vs true (green) landmark positions', figsize=(11, 8))
print('Green circle = true landmark, red dot = image point projected via H. '
      'They should coincide.')

## Next
If the gate passed and Views A/B/C look right, tell me and I'll build the
minimap (Phase 6: project actual player **foot points** → `outputs/minimap_frame.png`)
and then the video overlay (Phase 7). **Do not proceed if reprojection failed.**
